# `dman` Dataset Manager — Practical Walkthrough

This notebook is a **self-contained, end-to-end tutorial** for the `dman` dataset manager.  
Everything runs locally — no large downloads needed. We generate a tiny synthetic dataset inline.

### What you'll learn
1. Install `dman` and initialise a catalog
2. Create a mini YOLO dataset with PIL-generated images
3. Import, list, and inspect datasets via the CLI
4. Load and iterate datasets with the Python SDK
5. Access and visualise annotations
6. Add new samples (update workflow)
7. Export to COCO format
8. Convert to HuggingFace / PyTorch datasets
9. Clean up

### Prerequisites
- Python 3.9+
- Rust toolchain (for building `dman`)
- `Pillow`, `matplotlib` — `pip install pillow matplotlib`

### Install `dman`
Run once from the repo root:
```bash
pip install maturin
pip install .   # builds the Rust extensions and installs the dman package
```

## Section 1 — Introduction & Setup

We set `DMAN_HOME` to `/tmp/dman_demo` so this notebook never touches your real `~/.dman` catalog.  
**This must be done before importing `dman`.**

Shell cells (`! dman ...`) inherit environment variables set in Python cells above, so `DMAN_HOME`
is automatically forwarded to every CLI invocation.

In [ ]:
import os
import sys

# Point dman at an isolated demo catalog — MUST happen before importing dman
os.environ["DMAN_HOME"] = "/tmp/dman_demo"

print(f"DMAN_HOME = {os.environ['DMAN_HOME']}")
print(f"Python     = {sys.version}")

In [ ]:
# Initialise the demo catalog  (creates /tmp/dman_demo/catalog.db)
! dman init

## Section 2 — Create a Synthetic YOLO Dataset

We build a minimal YOLO dataset entirely in Python:
- **5 images** — 32 × 32 PNGs, each a solid colour (easy to inspect)
- **5 label files** — one bounding box per image (YOLO normalised format)
- **`data.yaml`** — YOLO dataset descriptor

The dataset has two classes: `cat` (id 0) and `dog` (id 1).

In [ ]:
import pathlib
from PIL import Image

YOLO_ROOT = pathlib.Path("/tmp/dman_yolo_demo")
IMG_DIR   = YOLO_ROOT / "images" / "train"
LBL_DIR   = YOLO_ROOT / "labels" / "train"

IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

# Five distinct colours for the five images
COLORS  = [(220, 80, 80), (80, 180, 80), (80, 120, 220), (220, 200, 60), (160, 80, 200)]
CLASSES = [0, 1, 0, 1, 0]   # alternating cat / dog

# Bounding boxes in normalised YOLO format: cx cy w h  (all in [0, 1])
BOXES   = [
    (0.50, 0.50, 0.40, 0.40),
    (0.30, 0.30, 0.30, 0.30),
    (0.60, 0.40, 0.35, 0.45),
    (0.45, 0.55, 0.50, 0.30),
    (0.55, 0.45, 0.25, 0.35),
]

for i, (color, cls, box) in enumerate(zip(COLORS, CLASSES, BOXES)):
    # ── image ────────────────────────────────────────────────────────────
    img = Image.new("RGB", (32, 32), color=color)
    img_path = IMG_DIR / f"img{i}.png"
    img.save(img_path)

    # ── label ────────────────────────────────────────────────────────────
    cx, cy, w, h = box
    label_path = LBL_DIR / f"img{i}.txt"
    label_path.write_text(f"{cls} {cx} {cy} {w} {h}\n")

# ── data.yaml ────────────────────────────────────────────────────────────
data_yaml = """path: .
train: images/train
nc: 2
names:
  0: cat
  1: dog
"""
(YOLO_ROOT / "data.yaml").write_text(data_yaml)

print(f"Synthetic YOLO dataset written to: {YOLO_ROOT}")
print("Structure:")
for p in sorted(YOLO_ROOT.rglob("*")):
    print(f"  {p.relative_to(YOLO_ROOT)}")

## Section 3 — Import via CLI

`dman import` reads the YOLO layout and registers the dataset in the catalog.  
We pass `--format yolo` explicitly and give it a friendly name with `--name`.

In [ ]:
! dman import /tmp/dman_yolo_demo --format yolo --name my_demo

## Section 4 — Browse Catalog via CLI

`dman list` shows every dataset in the catalog.  
`dman inspect <name>` shows sample/asset counts and format metadata.

In [ ]:
! dman list

In [ ]:
! dman inspect my_demo

## Section 5 — Load Dataset with the Python SDK

`dman.load_typed()` returns a `TypedDataset` — a lightweight, typed wrapper over the SQLite catalog.  
No data is copied; asset file paths point to the original files.

The `TypedDataset` provides `typed_samples()` and `typed_annotations()` methods that return proper
Python objects (`Sample`, `Asset`, `Annotation`, `BBox`) instead of raw dicts.

In [ ]:
import dman
from dman.types import Sample, Annotation, BBox

ds = dman.load_typed("my_demo")

print(f"Dataset name  : {ds.name}")
print(f"Dataset id    : {ds.dataset_id}")
print(f"Sample count  : {ds.sample_count()}")
print(f"Asset count   : {ds.asset_count()}")
print(f"len(dataset)  : {len(ds)}")

## Section 6 — Iterate Samples

`ds.typed_samples()` returns a list of `Sample` objects.  
Each `Sample` has an `assets` list of `Asset` objects — we filter by `asset_type` and read `file_path` directly.

In [ ]:
for s in ds.typed_samples():
    img_paths = [a.file_path for a in s.assets if a.asset_type == "image"]
    print(f"  Sample '{s.name}'  id={s.id}  images={img_paths}")

In [ ]:
# Indexing also works: ds[idx] returns the underlying raw dict structure.
# For user code, prefer typed_samples() shown above.
first = ds[0]
print("First sample keys :", list(first.keys()))
print("Assets             :", [{k: a[k] for k in ('asset_type','file_name')} for a in first['assets']])

## Section 7 — Access Annotations

`ds.typed_annotations(sample_name)` returns a list of `Annotation` objects.  

The `bbox` field is a `BBox` object with `.x`, `.y`, `.width`, `.height` attributes — no `json.loads()` needed.

In [ ]:
# Category id → name lookup  (populated by the YOLO importer)
CLASS_NAMES = {0: "cat", 1: "dog"}

for s in ds.typed_samples():
    for ann in ds.typed_annotations(s.name):
        if ann.bbox:
            print(f"  [{s.name}]  class={CLASS_NAMES.get(ann.category_id, f'class_{ann.category_id}')}  bbox=({ann.bbox.x:.1f}, {ann.bbox.y:.1f}, {ann.bbox.width:.1f}, {ann.bbox.height:.1f})")

## Section 8 — Visualise a Sample

We open the first sample's image with PIL, overlay the bounding box with matplotlib, and save the figure.  
Because the images are 32 × 32 pixels we scale them up for display.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# ── pick the first sample that has both an image and an annotation ────────
target_sample = ds.typed_samples()[0]
target_name   = target_sample.name

# ── find image asset ──────────────────────────────────────────────────────
img_asset = next(
    (a for a in target_sample.assets if a.asset_type == "image"),
    None,
)
if img_asset is None:
    raise RuntimeError(f"No image asset for sample '{target_name}'")

# ── load annotation ───────────────────────────────────────────────────────
anns = ds.typed_annotations(target_name)

# ── plot ──────────────────────────────────────────────────────────────────
img = Image.open(img_asset.file_path).resize((256, 256), Image.NEAREST)
img_w, img_h = img.size

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(np.array(img))
ax.set_title(f"Sample: {target_name}")
ax.axis("off")

for ann in anns:
    if ann.bbox is None:
        continue

    # Scale from original 32×32 to display 256×256
    scale_x = img_w / 32
    scale_y = img_h / 32

    rect = patches.Rectangle(
        (ann.bbox.x * scale_x, ann.bbox.y * scale_y),
        ann.bbox.width  * scale_x,
        ann.bbox.height * scale_y,
        linewidth=2, edgecolor="lime", facecolor="none",
    )
    ax.add_patch(rect)

    cat_name = CLASS_NAMES.get(ann.category_id, str(ann.category_id))
    ax.text(
        ann.bbox.x * scale_x, ann.bbox.y * scale_y - 4,
        cat_name, color="lime", fontsize=9,
        bbox=dict(facecolor="black", alpha=0.5, pad=1),
    )

out_path = "/tmp/dman_demo_vis.png"
plt.savefig(out_path, bbox_inches="tight", dpi=96)
plt.show()
print(f"Saved visualisation → {out_path}")

## Section 9 — Update the Dataset

`dman.update_dataset()` returns an **updater** that lets you add or remove samples without re-importing.  
We add a brand-new synthetic sample with an image and an annotation, then call `.apply()`.

In [ ]:
# Create one extra image to add
extra_img_path = "/tmp/dman_extra_img.png"
Image.new("RGB", (32, 32), color=(255, 160, 0)).save(extra_img_path)

# ── updater workflow ──────────────────────────────────────────────────────
updater = dman.update_dataset("my_demo")

updater.add_sample("extra_sample", metadata={"source": "manual"})

updater.add_asset(
    sample_name="extra_sample",
    asset_type="image",
    file_path=extra_img_path,
    width=32,
    height=32,
)

# We need the sample_id to add the annotation — fetch it after apply()
updater.apply()

print("Update applied.")

# ── reload and verify ─────────────────────────────────────────────────────
ds = dman.load_typed("my_demo")
print(f"Sample count after update: {ds.sample_count()}  (was 5, now should be 6)")

In [ ]:
# Add an annotation to the new sample using a second updater pass
extra = ds.get_sample("extra_sample")
if extra is not None:
    updater2 = dman.update_dataset("my_demo")
    updater2.add_annotation(
        sample_id=extra["id"],
        category="cat",
        bbox=[4, 4, 24, 24],   # [x, y, width, height] in pixels
    )
    updater2.apply()
    print("Annotation added to extra_sample.")

    # Confirm using typed API
    ds   = dman.load_typed("my_demo")
    anns = ds.typed_annotations("extra_sample")
    print(f"Annotations on extra_sample: {len(anns)}")
    for ann in anns:
        if ann.bbox:
            print(f"  bbox=({ann.bbox.x}, {ann.bbox.y}, {ann.bbox.width}, {ann.bbox.height})  category_id={ann.category_id}")

## Section 10 — Export to COCO Format

`dman export` converts any registered dataset to a standard format.  
Here we export `my_demo` as COCO JSON and pretty-print the result.

In [ ]:
import json
import shutil

COCO_OUT = "/tmp/dman_coco_output"
shutil.rmtree(COCO_OUT, ignore_errors=True)  # clean slate

! dman export my_demo /tmp/dman_coco_output --format coco

In [ ]:
# Find and pretty-print the COCO JSON file
coco_files = sorted(pathlib.Path(COCO_OUT).rglob("*.json"))
print(f"Exported files: {[str(f) for f in coco_files]}")

if coco_files:
    coco_data = json.loads(coco_files[0].read_text())

    print(f"\n=== COCO categories ===")
    for cat in coco_data.get("categories", []):
        print(f"  {cat}")

    print(f"\n=== COCO images (first 3) ===")
    for img in coco_data.get("images", [])[:3]:
        print(f"  {img}")

    print(f"\n=== COCO annotations (first 3) ===")
    for ann in coco_data.get("annotations", [])[:3]:
        print(f"  {ann}")

    print(f"\nTotal images      : {len(coco_data.get('images', []))}")
    print(f"Total annotations : {len(coco_data.get('annotations', []))}")

## Section 11 — HuggingFace & PyTorch Integration

`dataset.to_hf_dataset()` and `dataset.to_torch_dataset()` are optional — they require
`datasets` and `torch` respectively.  
We wrap each in a `try/except` so the notebook still runs without those packages.

In [ ]:
# ── HuggingFace datasets ──────────────────────────────────────────────────
try:
    hf_ds = ds.to_hf_dataset()
    print("HuggingFace dataset:", hf_ds)
    print("First row          :", hf_ds[0])
except ImportError:
    print("[skip] `datasets` package not installed — pip install datasets")
except Exception as e:
    print(f"[skip] HuggingFace conversion failed: {e}")

In [ ]:
# ── PyTorch dataset ───────────────────────────────────────────────────────
try:
    torch_ds = ds.to_torch_dataset()
    print("PyTorch dataset length:", len(torch_ds))
    print("First item keys       :", list(torch_ds[0].keys()) if hasattr(torch_ds[0], 'keys') else type(torch_ds[0]))
except ImportError:
    print("[skip] `torch` package not installed — pip install torch")
except Exception as e:
    print(f"[skip] PyTorch conversion failed: {e}")

## Section 12 — Cleanup

Remove the demo dataset from the catalog and delete the temporary directories.

In [ ]:
# Remove from catalog (the -y flag skips the confirmation prompt)
! dman remove my_demo -y

In [ ]:
# Confirm catalog is now empty
! dman list

In [ ]:
# Delete temp directories
for path in ["/tmp/dman_yolo_demo", "/tmp/dman_coco_output", "/tmp/dman_demo"]:
    shutil.rmtree(path, ignore_errors=True)
    print(f"Removed {path}")

extra_img = pathlib.Path("/tmp/dman_extra_img.png")
if extra_img.exists():
    extra_img.unlink()
    print(f"Removed {extra_img}")

vis_img = pathlib.Path("/tmp/dman_demo_vis.png")
if vis_img.exists():
    vis_img.unlink()
    print(f"Removed {vis_img}")

print("\nAll done! Catalog and temp files cleaned up.")

---

## Summary

| Step | What we did |
|------|-------------|
| 1 | Set `DMAN_HOME`, ran `! dman init` |
| 2 | Generated 5 synthetic 32×32 PNG images + YOLO labels with PIL |
| 3 | Imported the YOLO dataset via `! dman import` |
| 4 | Browsed the catalog with `! dman list` / `! dman inspect` |
| 5 | Loaded a `TypedDataset` with `dman.load_typed()` |
| 6 | Iterated typed `Sample` objects and their image file paths |
| 7 | Accessed bbox annotations directly via `ann.bbox.x` / `ann.bbox.width` |
| 8 | Visualised a sample with matplotlib + PIL |
| 9 | Added a new sample + annotation via `dman.update_dataset()` |
| 10 | Exported to COCO JSON with `! dman export` |
| 11 | Optionally converted to HuggingFace / PyTorch datasets |
| 12 | Removed the dataset and cleaned up temp files |

### Next steps
- Try with a real dataset: `dman import <your_path> --format yolo --name mydata`
- Explore `--format coco` and `--format huggingface` imports
- Define a custom schema YAML for typed metadata validation
- Write a Python plugin (`$DMAN_HOME/plugins/`) to support a custom format
- Serve the catalog over HTTP: `dman server start`